# Time-Series Classification Using Topological Features  

In this notebook we will investigate the use of persistent homology features for time-series classification, and compare them to conventional statistical features.

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root  / "src"))

from tda_features import delay_embed, extract_H1_features


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Construction of a Classification Dataset  

We have seen in the previous notebook that topological features can distinguish a periodic signal from a shuffled collection of the corresponding observations.  

Next, we want to investigate whether the five normalized H1 features indicated at the end of the previosu notebook can be used to systematically distinguish these two using supervised machine learning classification methods.  
We construct a dataset of paired signals. At each repetition a noisy sine signal is generated, and then a corresponding shuffled signal is constructed with the same observations.   
As seen before, the two signals inside of each pair have the same empirical distribution (mean and variance in particular), but different temporal structure.  

The following parameters are used:  
- Number of paired realizations: $100$;
- Numer of observations per signal: $500$;
- Gaussian noise standard deviation: $\sigma = 0.2$;
- Delay parameter: $\tau = 25$;
- Embedding dimension: $d = 3$;
- Prominence treshold: $0.1$

For each generated signal, the corresponding delay embedding is generated, and from that the persistent homology features are extracted.  
Each observation is assigned a class label (0 for shuffled, 1 for periodic), and a pair id is assigned to each pair of signals.

In [2]:
t = np.linspace(0, 10 * np.pi, 500, endpoint = False)
clean_signal = np.sin(t)

In [3]:
n_pairs = 100
rows = []
sigma = 0.2

rng = np.random.default_rng(2408)

for pair_id in range(n_pairs):

    noise = rng.normal(0, sigma, size = len(clean_signal))

    periodic_signal = clean_signal + noise
    shuffled_signal = rng.permutation(periodic_signal)

    periodic_embedding = delay_embed(periodic_signal, tau=25, d=3)
    shuffled_embedding = delay_embed(shuffled_signal, tau=25, d=3)

    periodic_features = extract_H1_features(periodic_embedding)
    shuffled_features = extract_H1_features(shuffled_embedding)
    
    periodic_features["label"] = 1
    periodic_features["pair_id"] = pair_id

    shuffled_features["label"] = 0
    shuffled_features["pair_id"] = pair_id

    rows.append(periodic_features)
    rows.append(shuffled_features)
    

In [4]:
dataframe = pd.DataFrame(rows)

print(dataframe.shape)
print(dataframe.head())

(200, 7)
     max_H1  second_H1  total_H1_persistence  prominent_H1_features  \
0  0.232153   0.040409              1.662104                      1   
1  0.067987   0.056519              2.882819                      0   
2  0.263603   0.039391              1.596972                      1   
3  0.062092   0.058250              2.990475                      0   
4  0.241457   0.031153              1.707876                      1   

   persistence_entropy  label  pair_id  
0             4.479697      1        0  
1             4.889494      0        0  
2             4.371877      1        1  
3             4.942093      0        1  
4             4.539097      1        2  


In [5]:
# Dataset structure
print(dataframe.shape)
print(dataframe["label"].value_counts())

# Pair integrity
pair_labels = dataframe.groupby("pair_id")["label"].agg(list)
print("Valid pairs:", pair_labels.apply(
    lambda labels: sorted(labels) == [0, 1]
).all())
print("Number of pairs:", len(pair_labels))

# Numerical validity
feature_cols = dataframe.columns.drop(["label", "pair_id"])
print("All features finite:",
      np.isfinite(dataframe[feature_cols].to_numpy()).all())

(200, 7)
label
1    100
0    100
Name: count, dtype: int64
Valid pairs: True
Number of pairs: 100
All features finite: True


## 2. Exploratory Analysis of Selected Topological Features  

We investigate how the  5 selected topological features differ between periodic and non-periodic (shuffled) signals.  

Before training a classifier, we will compare the mean, standard deviation and median of the feaures across the two classes of signals. We will use the paired structure of the dataser to directly observe the differences between each periodic signal, and its shuffled counterpart.  

We want to asses the discriminative potential, and stability, of the topological features, and determine whether the patterns observed in notebook two persist across multiple realizations.  

### 2.1 Descriptive and Paired Analysis of Periodic-Shuffled Signals

In [6]:
feature_cols = dataframe.columns.drop(["label", "pair_id"])

summary = (dataframe.groupby("label")[feature_cols].agg(["mean", "std", "median"]))

display(summary)

max_H1                     second_H1                      \
           mean       std    median      mean       std    median   
label                                                               
0      0.073989  0.008329  0.073503  0.067597  0.006522  0.067463   
1      0.248254  0.029848  0.251881  0.036865  0.008821  0.035097   

      total_H1_persistence                     prominent_H1_features       \
                      mean       std    median                  mean  std   
label                                                                       
0                 3.019419  0.223841  3.019505                  0.00  0.0   
1                 1.609846  0.125041  1.596048                  1.01  0.1   

             persistence_entropy                      
      median                mean       std    median  
label                                                 
0        0.0            4.849641  0.070490  4.850621  
1        1.0            4.418939  0.093407  4.412781

In [7]:
paired = dataframe.pivot( index = "pair_id", columns = "label", values = feature_cols) #Put each pair of signals on a row

differences = (paired.xs(1, level = "label", axis = 1)#Select columns of label 1 (periodic)
              - paired.xs(0, level = "label", axis = 1))#Selects columns of label 0 (shuffled)

differences_summary = pd.DataFrame({ #Summary statistics of pairwise differences
    "mean_difference": differences.mean(), #Equivalent to difference of means
    "std_difference": differences.std(), #Shows variability among paired differences
    "fraction_positive": (differences>0).mean()
})

display(differences_summary)

,mean_difference,std_difference,fraction_positive
max_H1,0.174264,0.029839,1.00
second_H1,-0.030732,0.011129,0.01
total_H1_persistence,-1.409573,0.248522,0.00
prominent_H1_features,1.010000,0.100000,1.00
persistence_entropy,-0.430702,0.116740,0.00


The five topological features we have selected exhibit some distinct patterns across the two groups of signal. The noisy **periodic** signal consistently showed a higher maximum H1 persistence feature, while the shuffled signals show higher total H1 persistence, as well as higher persistence entropy, together with a more significant second H1 feature, on average.  

Comparing each periodic signal with its shuffled counterpart shows that the differences are consistent among the 100 paired realizations.  
As can be seen in the table the results are consistent with periodic signals producing a strong, prominent H1 loop in their delay embedding, while the shuffled signals exhibit less concentrated, and more noise-like, persistent H1 features. The greater persistence entropy, and total H1 persistence in the shuffled signals, in particular, seems to be indicative of many short-lived H1 features, each with low individual "contributions", increasing those to features.  

The precise explenation for the differences in these last two features, however, would require more in-depth investigations of the persistence diagrams.

In [8]:
ranges = dataframe.groupby("label")[feature_cols].agg(["min", "max"]) #Checks range of each feature, for the two labels, 
                                                                      #to see if there is overlap
display(ranges)

display(pd.crosstab(dataframe["prominent_H1_features"], dataframe["label"])) #Frequency table of number of prominent H1 features for
                                                                             #each label

max_H1           second_H1           total_H1_persistence            \
            min       max       min       max                  min       max   
label                                                                          
0      0.057305  0.097003  0.054720  0.085969             2.526164  3.550060   
1      0.156600  0.305611  0.026705  0.105522             1.311085  1.958744   

      prominent_H1_features     persistence_entropy            
                        min max                 min       max  
label                                                          
0                         0   0            4.707605  5.015473  
1                         1   2            4.218427  4.638424

label,0,1
prominent_H1_features,,
0,100,0
1,0,99
2,0,1


We see that with the synthetic dataset, four of the five features have ranges that do not overlap:  
- Maximum normalized H1 persistence;
- Total H1 persistence;
- Number of prominent H1 feature (with threshold $0.1$);
- Persistence entropy.
The second-largest H1 features does overlap across the two classes.

We can thus see that, for this dataset, for four out ot of the five features we could set a threshold perfectly separating the two classes.  

**Limitations:**  
While these results exhibit perfect empirical separation for 4 features, there si no resaon to assume that this classification performance will generalize to different, more practical and realistic datasets. All the signals in this dataset where generated with the same underlying periodic structure, noise level, and embedding parameters.  

# 3. 